# Modell zur Ausgabenkategorisierung

Hybrider Ansatz aus **Speicher**, **Regeln** und einem **lokalen LLM** (`qwen2.5:3b` über Ollama).
Ziel: präzise Kategorien bei möglichst wenigen LLM-Abfragen. Alle Daten bleiben lokal.

**Pipeline pro Transaktion:**

1. **Normalisieren** – Payee wird bereinigt (Kleinschreibung, Umlaute, Zahlen, Satzzeichen, URLs, Ortsnamen).
2. **Speicher** – Treffer in der SQLite-Tabelle bekannter Händler? → Kategorie übernehmen.
3. **Regeln** – sonst: Schlüsselwortregeln (Supermärkte, Streaming, Verkehr, ...).
4. **LLM** – sonst: lokales LLM ordnet die Transaktion einer Kategorie zu.
5. **Speichern** – das Ergebnis landet im Speicher und wird beim nächsten Mal direkt wiederverwendet.

Manuelle Korrekturen überschreiben den Speichereintrag, sodass sich das System an das
individuelle Ausgabeverhalten anpasst.

## 1. Setup

Modell vorab laden: `ollama pull qwen2.5:3b`

`ExpenseDatabase` (in `Expanse_Class.py`) kapselt SQLite: Kategorien, Keyword-Regeln,
Händler-Speicher und Korrekturen. Kategorien und Keywords werden einmalig gesetzt – dafür
`init_categories.py` anpassen und im Terminal ausführen:

```bash
python init_categories.py
```
oder
```bash
python3 init_categories.py
```

`city_names` enthält alle Städtenamen (akzentfrei, klein) und dient später zum Herausfiltern
von Ortsangaben aus dem Payee.

In [1]:
import json
import re
import sys
import unicodedata
from pathlib import Path
from string import Template

import geonamescache
import ollama
import pandas as pd

from Expanse_Class import ExpenseDatabase

# DKB-Parser aus dem Backend, um echte Testdaten zu laden
sys.path.append("../../")
from backend.parsers import DKB

gc = geonamescache.GeonamesCache()

city_names = {
    "".join(
        c for c in unicodedata.normalize("NFKD", city["name"].lower())
        if not unicodedata.combining(c)
    )
    for city in gc.get_cities().values()
}

## 2. Textnormalisierung

`clean_payee` macht aus verschiedenen Schreibweisen desselben Händlers einen einheitlichen Schlüssel,
z. B. `ALDI SUED FIL.017 KIEL` → `aldi sued fil`.

> ⚠️ Der letzte Schritt entfernt **jedes** Wort, das ein Städtename ist. Marken, die wie eine
> (auch ausländische) Stadt heißen, können dadurch verloren gehen.

In [2]:
def clean_payee(text: str) -> str:
    """
    Normalisiert einen Payee-String:
    Kleinschreibung, ohne Akzente, URLs, alleinstehende Zahlen,
    Sonderzeichen, Satzzeichen, Ortsnamen und Mehrfach-Leerzeichen.
    """
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # Akzente entfernen (ä -> a, é -> e, ...)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    text = re.sub(r"\.com\b|\.co\.uk\b|www\.", " ", text)  # URLs / Domains
    text = re.sub(r"\b\d+\b", " ", text)                    # alleinstehende Zahlen
    text = re.sub(r"[*#]", " ", text)                       # Sonderzeichen
    text = re.sub(r"[^\w\s]", " ", text)                    # Satzzeichen
    text = re.sub(r"\s+", " ", text).strip()                # Leerzeichen

    # Ortsnamen entfernen
    return " ".join(w for w in text.split() if w not in city_names)

## 3. LLM-Fallback

Wird nur aufgerufen, wenn weder Speicher noch Regeln greifen.

Der Prompt liegt in `prompts/classify_transaction.txt` und wird einmalig geladen. Er gibt dem
Modell drei Dinge vor: Zahlungsdienstleister und Banking-Rauschen ignorieren, den **echten
Händler** kategorisieren (`PAYPAL *SPOTIFY` → `abos`, nicht PayPal) und die Antwort als striktes
JSON zurückgeben (`format="json"`).

Platzhalter im Template: `$categories` (kommt aus `db.build_prompt()`) und `$description`.
Wir nutzen `string.Template` statt f-String, damit die JSON-Klammern im Prompt nicht
verdoppelt werden müssen.

In [3]:
PROMPT_TEMPLATE = Template(
    Path("prompts/classify_transaction.txt").read_text(encoding="utf-8")
)


def classify_with_llm(description, db):
    prompt = PROMPT_TEMPLATE.substitute(
        categories=db.build_prompt(),
        description=description,
    )

    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}],
        format="json",
    )

    result = json.loads(response["message"]["content"])

    return {
        "category": result["category"],
        "confidence": float(result["confidence"]),
        "source": "llm",
    }

## 4. Klassifikations-Pipeline

`classify_expenses` läuft über alle Zeilen und probiert der Reihe nach: Speicher → Regeln → LLM.
Das Ergebnis wird unter dem normalisierten Händlernamen gespeichert und landet in drei neuen Spalten:
`category`, `confidence`, `classification_source`.

In [4]:
def classify_expenses(df):
    db = ExpenseDatabase()
    results = []

    for _, row in df.iterrows():
        description = row["classifier"]

        result = (
            db.lookup_memory(description)
            or db.apply_rules(description)
            or classify_with_llm(clean_payee(description), db)
        )

        # Ergebnis für zukünftige Transaktionen merken
        db.save_memory(clean_payee(description), result["category"], result["confidence"])
        results.append(result)

    output = df.copy()
    output["category"] = [r["category"] for r in results]
    output["confidence"] = [r["confidence"] for r in results]
    output["classification_source"] = [r["source"] for r in results]

    db.close()
    return output

## 5. Manuelle Korrektur

Überschreibt den Speichereintrag eines Händlers. Alle künftigen Transaktionen desselben
Händlers erhalten automatisch die korrigierte Kategorie.

In [5]:
def correct_transaction(df, index, new_category, db):
    """Korrigiert eine Transaktion manuell und merkt sich die Kategorie."""
    merchant = clean_payee(df.loc[index, "classifier"])
    db.correct_merchant(merchant, new_category)

## 6. Test der Pipeline

Synthetische, bereits kategorisierte DKB-Daten. `classifier` ist der Text, den die Pipeline sieht
(Payee + Verwendungszweck). Aus der wahren Kategorie wird die führende Nummerierung (`1. `) entfernt.

In [7]:
transactions = DKB.load("../../data/dkb_synth_large.csv")

transactions["classifier"] = transactions["payee"] + " " + transactions["description"]
transactions["category"] = transactions["category"].str.replace(r"^\d+\.\s*", "", regex=True)

result = classify_expenses(transactions.head(50))

print(result[["classifier", "category", "confidence", "classification_source"]].head(20))

                                           classifier       category  \
0                     Lucky Luke Beitrag zum Geschenk  lifestyle&fun   
1            1und1 DSL Rechnung 06/2026 Kd-Nr. 059784                  
2   PayPal Europe S.a.r.l. et Cie S.C.A 9946418398...       mobility   
3   Deutsche Glasfaser Rechnung 10/2025 Kd-Nr. 951...       mobility   
4   Spargelhof.Klein.0047/Düsseldorf SPARGELHOF SA...      groceries   
5   Rossmann Babywelt Lastschrift 10/2025 Ref 1705...         family   
6         wilhelm.tel Rechnung 06/2026 Kd-Nr. 7801104                  
7           KVB Köln Lastschrift 09/2025 Ref 79736383                  
8   PayPal Europe S.a.r.l. et Cie S.C.A 6355373096...       mobility   
9   PayPal Europe S.a.r.l. et Cie S.C.A 1584668793...     healthcare   
10  Hotel.Astoria.124/Köln HOTEL 2025-08-14T16:29:...  lifestyle&fun   
11  PayPal Europe S.a.r.l. et Cie S.C.A 4510173973...      groceries   
12  PayPal Europe S.a.r.l. et Cie S.C.A 6198273481...  subscript

## 7. Vergleich: Vorhersage vs. tatsächliche Kategorie


In [8]:
comparison = result.merge(
    transactions[["classifier", "category"]],
    on="classifier",
    suffixes=("_predicted", "_true"),
)

comparison["correct"] = comparison["category_predicted"] == comparison["category_true"]

correct = comparison["correct"].sum()
total = len(comparison)

print(f"Accuracy: {comparison['correct'].mean():.2%}")
print(f"Correct: {correct}/{total}")
print(f"Wrong:   {total - correct}/{total}")

Accuracy: 42.00%
Correct: 21/50
Wrong:   29/50
